# LTR Model Comparison — Pointwise vs RankNet vs LambdaRank
### Statistical comparison across all three Learning-to-Rank modes
---
**Dataset**: LETOR4 (MQ2008)  
**Evaluation**: 5-Fold (Standard LETOR Folds), 3 seeds each  
**Significance**: Paired t-test (Fix #7)

> **Note**: Run notebooks `01_pointwise.ipynb`, `02_ranknet.ipynb`, and
> `03_lambdarank.ipynb` first to generate the results files loaded here.


## Step 1 · Setup & Load Pre-computed Results

In [ ]:
# ── Colab Setup ───────────────────────────────────────────────────────────────
# This cell clones the repo (if needed) and installs the ltr package.
# Change REPO_PATH if you cloned to a different location.
import os, subprocess, sys

REPO_PATH = "/content/Learning-To-Rank-for-Search"

if not os.path.exists(REPO_PATH):
    subprocess.run(
        ["git", "clone",
         "https://github.com/navaneeswar854/Learning-To-Rank-for-Search.git",
         REPO_PATH],
        check=True,
    )
    print("Repo cloned.")
else:
    print("Repo already present.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", REPO_PATH + "/LTR", "-q"],
    check=True,
)
print("ltr package installed.")

# Add to sys.path so the kernel finds it immediately without restarting
sys.path.insert(0, REPO_PATH + "/LTR/src")


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ltr.metrics import paired_significance

RESULTS_DIR = "/content/ltr_results"
K_LIST = [1, 3, 5, 10]

def load_results(mode: str) -> dict:
    path = f"{RESULTS_DIR}/{mode}_results.json"
    with open(path) as f:
        return json.load(f)

pointwise_results   = load_results("pointwise")
ranknet_results     = load_results("ranknet")
lambdarank_results  = load_results("lambdarank")

print("Results loaded successfully.")
print(f"  Pointwise   folds: {len(pointwise_results['fold_results'])}")
print(f"  RankNet     folds: {len(ranknet_results['fold_results'])}")
print(f"  LambdaRank  folds: {len(lambdarank_results['fold_results'])}")


## Step 2 · Summary Table — Mean ± Std Across All Folds × Seeds

In [ ]:
def print_comparison_table(results_dict: dict, k_list=None):
    """Print a comparison table with Mean ± Std for each model and k."""
    if k_list is None:
        k_list = K_LIST

    models = {
        "Pointwise":  results_dict["pointwise"],
        "RankNet":    results_dict["ranknet"],
        "LambdaRank": results_dict["lambdarank"],
    }

    print(f"\n{'═'*70}")
    print(f"  MODEL COMPARISON — NDCG (Mean ± Std over 5 folds × 3 seeds)")
    print(f"{'═'*70}")
    header = f"{'Metric':<10}"
    for name in models:
        header += f"  {name:>20}"
    print(header)
    print(f"{'─'*70}")

    for k in k_list:
        row = f"NDCG@{k:<5}"
        for name, res in models.items():
            # overall is keyed by string k (JSON) or int k
            overall = res.get("overall", {})
            entry = overall.get(str(k), overall.get(k, {"mean": 0.0, "std": 0.0}))
            row += f"  {entry['mean']:.4f} ± {entry['std']:.4f}   "
        print(row)
    print(f"{'═'*70}")

results_dict = {
    "pointwise":  pointwise_results,
    "ranknet":    ranknet_results,
    "lambdarank": lambdarank_results,
}

print_comparison_table(results_dict)


## Step 3 · Visualisation — NDCG@k by Model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_info = {
    "Pointwise":  (pointwise_results,  "steelblue"),
    "RankNet":    (ranknet_results,     "tomato"),
    "LambdaRank": (lambdarank_results,  "seagreen"),
}

# ── Left: Bar chart of NDCG@10 ───────────────────────────────────────────────
ax = axes[0]
names  = list(models_info.keys())
means  = []
stds   = []
colors = []

for name, (res, color) in models_info.items():
    overall = res.get("overall", {})
    entry   = overall.get("10", overall.get(10, {"mean": 0.0, "std": 0.0}))
    means.append(entry["mean"])
    stds.append(entry["std"])
    colors.append(color)

bars = ax.bar(names, means, yerr=stds, color=colors, alpha=0.8,
              capsize=6, edgecolor="white", linewidth=1.2)
ax.set_ylabel("NDCG@10")
ax.set_title("NDCG@10 Comparison (Mean ± Std)", fontweight="bold")
ax.set_ylim(0, 1)
ax.grid(axis="y", linestyle="--", alpha=0.5)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{mean:.4f}", ha="center", va="bottom", fontsize=10)

# ── Right: Line chart across k values ────────────────────────────────────────
ax = axes[1]
for name, (res, color) in models_info.items():
    overall = res.get("overall", {})
    y_means = []
    y_stds  = []
    for k in K_LIST:
        entry = overall.get(str(k), overall.get(k, {"mean": 0.0, "std": 0.0}))
        y_means.append(entry["mean"])
        y_stds.append(entry["std"])
    ax.plot(K_LIST, y_means, marker="o", label=name, color=color, linewidth=2)
    ax.fill_between(K_LIST,
                    [m - s for m, s in zip(y_means, y_stds)],
                    [m + s for m, s in zip(y_means, y_stds)],
                    alpha=0.15, color=color)

ax.set_xlabel("k")
ax.set_ylabel("NDCG@k")
ax.set_title("NDCG@k Across Cutoffs", fontweight="bold")
ax.set_xticks(K_LIST)
ax.legend()
ax.grid(linestyle="--", alpha=0.5)

plt.suptitle("LTR Model Comparison — LETOR4 MQ2008", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## Step 4 · Statistical Significance Testing (Fix #7)

We use a **paired t-test** (`scipy.stats.ttest_rel`) on the raw per-query NDCG@10
arrays from the 5-fold evaluation.  Because each fold produces a fixed set of test
queries evaluated by all models, the data is naturally paired.

A **p-value < 0.05** indicates the difference between two models is statistically
significant at the 5% level.


In [ ]:
# Flatten per-query NDCG@10 values from all folds and seeds
def extract_per_query_ndcg10(results: dict, k: int = 10) -> list:
    """
    Extract a flat list of per-seed mean NDCG@k values across all folds.
    These serve as the paired observations for the t-test.
    """
    values = []
    for fold_res in results.get("fold_results", []):
        for seed_res in fold_res.get("per_seed", []):
            key = str(k) if str(k) in seed_res else k
            values.append(seed_res.get(key, seed_res.get(str(key), 0.0)))
    return values

pw_scores  = extract_per_query_ndcg10(pointwise_results,  k=10)
rn_scores  = extract_per_query_ndcg10(ranknet_results,    k=10)
lr_scores  = extract_per_query_ndcg10(lambdarank_results,  k=10)

print(f"Observations per model: {len(pw_scores)} (folds × seeds)")
print()

pairs = [
    ("Pointwise",  "RankNet",    pw_scores,  rn_scores),
    ("Pointwise",  "LambdaRank", pw_scores,  lr_scores),
    ("RankNet",    "LambdaRank", rn_scores,  lr_scores),
]

print(f"{'─'*55}")
print(f"{'Comparison':<30}  {'p-value':>10}  {'Significant?':>12}")
print(f"{'─'*55}")
for name_a, name_b, a, b in pairs:
    p = paired_significance(a, b)
    sig = "✓ Yes" if p < 0.05 else "✗ No"
    label = f"{name_a} vs {name_b}"
    print(f"{label:<30}  {p:>10.4f}  {sig:>12}")
print(f"{'─'*55}")
print("(α = 0.05, two-tailed paired t-test)")


## Step 5 · Per-Fold NDCG@10 Summary

In [ ]:
import pandas as pd

rows = []
for mode, results in [("Pointwise", pointwise_results),
                       ("RankNet",   ranknet_results),
                       ("LambdaRank",lambdarank_results)]:
    for fold_res in results.get("fold_results", []):
        fold_num = fold_res["fold"]
        summary  = fold_res["summary"]
        key = "10" if "10" in summary else 10
        entry = summary.get(key, {"mean": 0.0, "std": 0.0})
        rows.append({
            "Model": mode,
            "Fold":  fold_num,
            "NDCG@10 Mean": round(entry["mean"], 4),
            "NDCG@10 Std":  round(entry["std"],  4),
        })

df = pd.DataFrame(rows).pivot(index="Fold", columns="Model", values="NDCG@10 Mean")
print(df.to_string())
print()
print("Overall Mean:")
print(df.mean().to_string())


## Conclusion

This notebook provides a fair, statistically grounded comparison of all three LTR approaches
on the standard LETOR4 / MQ2008 benchmark:

- **Same network** (`ScoringMLP`) for all three modes — architecture is not a confound.
- **Same 5 author-provided folds** — results are reproducible and comparable to papers.
- **3 random seeds per fold** — variance is quantified, not hidden.
- **Paired t-test** — significance of observed differences is formally tested.
